# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamKottish/FlyRankML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

import os
import getpass
import duckdb
import pandas as pd
import numpy as np
import sklearn

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

# Hugging Face authentication
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token: "
)

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_MAR = (
    f"read_parquet('{REL}/"
    "fact_content_daily_performance/month=2026-03/*.parquet')"
)

FACT_APR = (
    f"read_parquet('{REL}/"
    "fact_content_daily_performance/month=2026-04/*.parquet')"
)

print("✓ Connected to FlyRank warehouse")
print("Feature window: March 2026")
print("Outcome window: April 2026")
print("Random seed:", RANDOM_STATE)
print("scikit-learn:", sklearn.__version__)

Paste your Hugging Face READ token: ··········
✓ Connected to FlyRank warehouse
Feature window: March 2026
Outcome window: April 2026
Random seed: 42
scikit-learn: 1.6.1


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


My lane is a **ranking/scoring problem**: the useful output is an ordered list of content items that an SEO specialist or content editor should review first.

My primary learned method is a **Random Forest classifier**. I use its predicted probability of the April CTR-opportunity proxy as the ranking score. Random Forest is appropriate because relationships among impressions, CTR, search position, position volatility, and activity may be nonlinear and may interact.

I also train **Logistic Regression** as a simpler learned comparison. This gives me a readable linear model before relying on the more flexible Random Forest.

Neither model receives April measurements as features. All predictive inputs come from March 2026. April 2026 is used only to construct the later observed decision-support proxy.

The model must beat the transparent Week-4 baseline on the same held-out clients and the same Precision@K metrics before additional complexity is considered useful.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# MARCH FEATURES
# ============================================================

march = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS feature_impressions,
        SUM(gsc_clicks) AS feature_clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks)
                 / SUM(gsc_impressions)
        END AS feature_ctr,

        SUM(
            CASE
                WHEN gsc_avg_position > 0
                     AND gsc_impressions > 0
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN gsc_avg_position > 0
                         AND gsc_impressions > 0
                    THEN gsc_impressions
                    ELSE 0
                END
            ),
            0
        ) AS feature_avg_position,

        STDDEV_SAMP(
            CASE
                WHEN gsc_avg_position > 0
                     AND gsc_impressions > 0
                THEN gsc_avg_position
            END
        ) AS feature_position_std,

        COUNT(
            DISTINCT CASE
                WHEN gsc_impressions > 0
                THEN report_date
            END
        ) AS feature_active_days

    FROM {FACT_MAR}

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(gsc_impressions) >= 500
""").df()

march["feature_position_std"] = (
    march["feature_position_std"].fillna(0)
)

# The Week-4 baseline requires measurable March position.
march = march[
    march["feature_avg_position"].notna()
].copy()


# ============================================================
# APRIL OUTCOME
# ============================================================

april = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS outcome_impressions,
        SUM(gsc_clicks) AS outcome_clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 100.0 * SUM(gsc_clicks)
                 / SUM(gsc_impressions)
        END AS outcome_ctr,

        SUM(
            CASE
                WHEN gsc_avg_position > 0
                     AND gsc_impressions > 0
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN gsc_avg_position > 0
                         AND gsc_impressions > 0
                    THEN gsc_impressions
                    ELSE 0
                END
            ),
            0
        ) AS outcome_avg_position

    FROM {FACT_APR}

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(gsc_impressions) >= 500
""").df()

april = april[
    april["outcome_avg_position"].notna()
].copy()


# ============================================================
# POSITION BAND
# ============================================================

def position_band(position):
    if position <= 3:
        return "top_3"
    elif position <= 10:
        return "page_1"
    elif position <= 20:
        return "striking"
    elif position <= 50:
        return "page_3_5"
    return "deep"


april["outcome_position_band"] = (
    april["outcome_avg_position"]
    .apply(position_band)
)

# Position-adjusted April peer comparison
april["outcome_band_median_ctr"] = (
    april
    .groupby("outcome_position_band")["outcome_ctr"]
    .transform("median")
)

april["outcome_ctr_gap_pp"] = (
    april["outcome_band_median_ctr"]
    - april["outcome_ctr"]
)

# Same provisional opportunity definition used previously
april["opportunity_proxy"] = (
    april["outcome_ctr_gap_pp"] > 0.10
).astype(int)


# ============================================================
# JOIN MARCH -> APRIL
# ============================================================

model_df = march.merge(
    april[
        [
            "client_hash_id",
            "content_hash_id",
            "outcome_impressions",
            "outcome_ctr",
            "outcome_avg_position",
            "outcome_position_band",
            "outcome_ctr_gap_pp",
            "opportunity_proxy",
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)


# Safe engineered March features
model_df["log_impressions"] = np.log1p(
    model_df["feature_impressions"]
)

model_df["log_clicks"] = np.log1p(
    model_df["feature_clicks"]
)


FEATURES = [
    "log_impressions",
    "log_clicks",
    "feature_ctr",
    "feature_avg_position",
    "feature_position_std",
    "feature_active_days",
]

TARGET = "opportunity_proxy"


print(f"Modeling rows: {len(model_df):,}")
print(
    f"Clients: "
    f"{model_df['client_hash_id'].nunique():,}"
)
print(
    f"April opportunity base rate: "
    f"{model_df[TARGET].mean():.1%}"
)

print("\nFeatures:")
for col in FEATURES:
    print("-", col)

# Leakage guard
assert not any(
    "outcome" in col.lower()
    or "april" in col.lower()
    or "proxy" in col.lower()
    for col in FEATURES
)

assert "client_hash_id" not in FEATURES
assert "content_hash_id" not in FEATURES

print("\n✓ Feature list contains March information only.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling rows: 51,496
Clients: 34
April opportunity base rate: 23.9%

Features:
- log_impressions
- log_clicks
- feature_ctr
- feature_avg_position
- feature_position_std
- feature_active_days

✓ Feature list contains March information only.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


I use a **grouped client holdout**.

Approximately 75% of the pseudonymized clients are used for training and 25% are held out for testing. `client_hash_id` is the grouping variable, so the same client can never appear in both sets.

This is more honest than randomly splitting individual pages because pages belonging to one client can share hidden characteristics such as topic mix, traffic scale, search behavior, and content strategy. A random page split could therefore exaggerate performance by letting the model recognize client-specific patterns.

The timeline is also time-aware: all features come from March 2026, while the opportunity proxy is measured in April 2026.

I use a fixed random seed of 42 so the split and model results are reproducible.

One limitation is that evaluation requires at least 500 April impressions so the CTR opportunity proxy has enough evidence. This uses future information to determine which rows are evaluable, but April information never enters the March score itself.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# GROUPED CLIENT SPLIT
# ============================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        model_df[TARGET],
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

train_clients = set(train_df["client_hash_id"])
test_clients = set(test_df["client_hash_id"])

overlap = train_clients.intersection(test_clients)

assert len(overlap) == 0

print(f"Training rows: {len(train_df):,}")
print(f"Test rows:     {len(test_df):,}")

print(
    f"\nTraining clients: "
    f"{len(train_clients)}"
)

print(
    f"Test clients:     "
    f"{len(test_clients)}"
)

print(
    f"Client overlap:   "
    f"{len(overlap)}"
)

print(
    f"\nTrain base rate: "
    f"{train_df[TARGET].mean():.3f}"
)

print(
    f"Test base rate:  "
    f"{test_df[TARGET].mean():.3f}"
)

print("\n✓ No client appears in both train and test.")


X_train = train_df[FEATURES].copy()
X_test = test_df[FEATURES].copy()

y_train = train_df[TARGET].copy()
y_test = test_df[TARGET].copy()

Training rows: 25,484
Test rows:     26,012

Training clients: 25
Test clients:     9
Client overlap:   0

Train base rate: 0.263
Test base rate:  0.215

✓ No client appears in both train and test.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I compare four references on the exact same held-out client set:

1. the April proxy base rate,
2. my transparent Week-4 position-adjusted CTR baseline,
3. Logistic Regression,
4. Random Forest.

The Week-4 baseline remains a hand-written score: a page receives more priority when its March CTR is below the typical March CTR of pages in the same position band and it has greater search exposure.

The learned models use the same March feature population and are evaluated against the same April proxy.

My primary metric is **Precision@50** because the practical question is whether the first pages sent to a reviewer are useful candidates. I also report Precision@20, average precision, and ROC-AUC for context.

I will not claim that ML is better unless the held-out comparison actually shows it.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# BASELINE
# ============================================================

# Fit the peer CTR reference using TRAIN clients only.
train_baseline = train_df.copy()
test_baseline = test_df.copy()

train_baseline["feature_position_band"] = (
    train_baseline["feature_avg_position"]
    .apply(position_band)
)

test_baseline["feature_position_band"] = (
    test_baseline["feature_avg_position"]
    .apply(position_band)
)

train_band_medians = (
    train_baseline
    .groupby("feature_position_band")["feature_ctr"]
    .median()
    .to_dict()
)

overall_train_ctr_median = (
    train_baseline["feature_ctr"].median()
)

test_baseline["band_median_ctr"] = (
    test_baseline["feature_position_band"]
    .map(train_band_medians)
    .fillna(overall_train_ctr_median)
)

test_baseline["baseline_ctr_gap_pp"] = (
    test_baseline["band_median_ctr"]
    - test_baseline["feature_ctr"]
)

test_baseline["positive_ctr_gap"] = (
    test_baseline["baseline_ctr_gap_pp"]
    .clip(lower=0)
)

# Same transparent Week-4 baseline formula
test_baseline["baseline_score"] = (
    test_baseline["positive_ctr_gap"]
    * np.log1p(test_baseline["feature_impressions"])
)

baseline_scores = test_baseline["baseline_score"].to_numpy()


# ============================================================
# LOGISTIC REGRESSION
# ============================================================

logistic = Pipeline([
    ("scale", StandardScaler()),
    (
        "model",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        )
    ),
])

logistic.fit(X_train, y_train)

logistic_scores = logistic.predict_proba(
    X_test
)[:, 1]


# ============================================================
# RANDOM FOREST
# ============================================================

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_scores = rf.predict_proba(
    X_test
)[:, 1]


# ============================================================
# METRICS
# ============================================================

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    order = np.argsort(-scores)

    return y_true[order[:k]].mean()


def safe_auc(y_true, scores):
    if len(np.unique(y_true)) < 2:
        return np.nan

    return roc_auc_score(y_true, scores)


def evaluate_ranking(name, y_true, scores):
    return {
        "method": name,
        "precision@20": precision_at_k(
            y_true, scores, 20
        ),
        "precision@50": precision_at_k(
            y_true, scores, 50
        ),
        "average_precision": average_precision_score(
            y_true, scores
        ),
        "roc_auc": safe_auc(
            y_true, scores
        ),
    }


results = []

# Random ranking expectation = base rate
results.append({
    "method": "base_rate",
    "precision@20": y_test.mean(),
    "precision@50": y_test.mean(),
    "average_precision": y_test.mean(),
    "roc_auc": 0.5,
})

results.append(
    evaluate_ranking(
        "week4_baseline",
        y_test,
        baseline_scores
    )
)

results.append(
    evaluate_ranking(
        "logistic_regression",
        y_test,
        logistic_scores
    )
)

results.append(
    evaluate_ranking(
        "random_forest",
        y_test,
        rf_scores
    )
)

results_df = pd.DataFrame(results)

metric_cols = [
    "precision@20",
    "precision@50",
    "average_precision",
    "roc_auc",
]

results_df[metric_cols] = (
    results_df[metric_cols].round(3)
)

display(results_df)

print(
    f"Test base rate: "
    f"{y_test.mean():.3f}"
)

print(
    "\nPrimary comparison = held-out Precision@50"
)

,method,precision@20,precision@50,average_precision,roc_auc
0,base_rate,0.215,0.215,0.215,0.500
1,week4_baseline,0.850,0.880,0.555,0.795
2,logistic_regression,0.800,0.860,0.567,0.850
3,random_forest,1.000,0.880,0.611,0.865


Test base rate: 0.215

Primary comparison = held-out Precision@50


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

I interpret the Random Forest as a ranking aid rather than an automatic editing decision.

First, I inspect the highest-ranked false positives: pages the model strongly prioritized but that did not meet the later April CTR-opportunity proxy. These cases matter because they would consume reviewer time without being supported by the chosen later proxy.

I also inspect missed opportunities: April-positive pages that received relatively low model scores.

Finally, I use permutation importance on the held-out clients to see which March features the model depends on. A strong feature is not automatically causal; it only means that changing or shuffling that observed feature reduces predictive ranking performance.

Potential errors may come from changes in search position between March and April, seasonality, SERP changes, low-volume variation, unobserved intent differences, or limitations in the rule-defined April proxy itself. The model therefore provides directional decision-support, not proof that editing a page will improve CTR.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# ADD PREDICTIONS TO HELD-OUT FRAME
# ============================================================

errors = test_df.copy()

errors["rf_score"] = rf_scores

errors["rf_predicted_class"] = (
    errors["rf_score"] >= 0.5
).astype(int)


# ============================================================
# TOP-50 RANKING ERRORS
# ============================================================

ranked_test = errors.sort_values(
    "rf_score",
    ascending=False
).reset_index(drop=True)

ranked_test["model_rank"] = (
    np.arange(1, len(ranked_test) + 1)
)

top50 = ranked_test.head(
    min(50, len(ranked_test))
).copy()

top50_false_positives = (
    top50[
        top50[TARGET] == 0
    ]
    .head(3)
)

print("Three high-ranked false positives:")

display(
    top50_false_positives[
        [
            "model_rank",
            "content_hash_id",
            "rf_score",
            "feature_impressions",
            "feature_ctr",
            "feature_avg_position",
            "feature_position_std",
            "outcome_ctr",
            "outcome_avg_position",
            "outcome_ctr_gap_pp",
            TARGET,
        ]
    ]
)


# ============================================================
# MISSED APRIL OPPORTUNITIES
# ============================================================

missed = (
    ranked_test[
        ranked_test[TARGET] == 1
    ]
    .sort_values(
        "rf_score",
        ascending=True
    )
    .head(3)
)

print("\nThree positive cases given relatively low scores:")

display(
    missed[
        [
            "model_rank",
            "content_hash_id",
            "rf_score",
            "feature_impressions",
            "feature_ctr",
            "feature_avg_position",
            "feature_position_std",
            "outcome_ctr",
            "outcome_avg_position",
            "outcome_ctr_gap_pp",
            TARGET,
        ]
    ]
)


# ============================================================
# PERMUTATION IMPORTANCE ON HELD-OUT CLIENTS
# ============================================================

perm = permutation_importance(
    rf,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values(
    "importance_mean",
    ascending=False
)

print("\nPermutation importance:")
display(importance_df.round(4))

print("\nTop 3 features:")
for feature in importance_df.head(3)["feature"]:
    print("-", feature)


# ============================================================
# ERROR RATE BY POSITION BAND
# ============================================================

ranked_test["march_position_band"] = (
    ranked_test["feature_avg_position"]
    .apply(position_band)
)

ranked_test["classification_error"] = (
    ranked_test["rf_predicted_class"]
    != ranked_test[TARGET]
).astype(int)

error_by_band = (
    ranked_test
    .groupby("march_position_band")
    .agg(
        n=("content_hash_id", "size"),
        opportunity_rate=(TARGET, "mean"),
        error_rate=("classification_error", "mean"),
        median_rf_score=("rf_score", "median"),
    )
    .sort_values("error_rate", ascending=False)
)

print("\nErrors by March position band:")
display(error_by_band.round(3))

Three high-ranked false positives:


,model_rank,content_hash_id,rf_score,feature_impressions,feature_ctr,feature_avg_position,feature_position_std,outcome_ctr,outcome_avg_position,outcome_ctr_gap_pp,opportunity_proxy
22,23,content_6667a5acd8f9c0aa,0.904519,7694.0,0.038991,7.068235,0.712891,0.123982,6.882572,0.085223,0
24,25,content_8b10a46669bfb27a,0.904175,11629.0,0.042996,4.945739,0.913722,0.158400,4.525493,0.050805,0
29,30,content_eb083d548c0bfe7d,0.903776,5605.0,0.017841,0.981088,0.759000,0.344828,2.207759,-0.151777,0



Three positive cases given relatively low scores:


,model_rank,content_hash_id,rf_score,feature_impressions,feature_ctr,feature_avg_position,feature_position_std,outcome_ctr,outcome_avg_position,outcome_ctr_gap_pp,opportunity_proxy
25036,25037,content_fbe69c754ffbebac,0.015146,2097.0,1.478302,16.448259,6.656344,0.000000,17.140468,0.163132,1
24791,24792,content_3539d16ff8091e01,0.017889,11656.0,1.132464,7.023679,3.263106,0.000000,10.765093,0.163132,1
24620,24621,content_e5b557741dd26f15,0.019540,25433.0,0.393190,2.471042,0.344865,0.098958,3.122707,0.110247,1



Permutation importance:


,feature,importance_mean,importance_std
3,feature_avg_position,0.2900,0.0038
2,feature_ctr,0.1996,0.0059
1,log_clicks,0.0477,0.0039
0,log_impressions,0.0427,0.0027
4,feature_position_std,0.0160,0.0018
5,feature_active_days,0.0005,0.0010



Top 3 features:
- feature_avg_position
- feature_ctr
- log_clicks

Errors by March position band:


,n,opportunity_rate,error_rate,median_rf_score
march_position_band,,,,
page_1,11680,0.319,0.322,0.576
striking,4782,0.201,0.283,0.383
top_3,3079,0.245,0.268,0.425
page_3_5,6291,0.025,0.025,0.085
deep,180,0.000,0.000,0.093


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.